# AirNow NO₂ — Extended Time Series Plots

Additional NO₂ visualisations built on the same AirNow dataset used in `01_explore_airnow.ipynb`.
All figures are saved to `../plots/`.

In [1]:
import sys, os
from pathlib import Path

_here = Path(os.getcwd()).resolve()
ROOT  = _here.parent if _here.name == 'notebooks' else _here
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

DATA_DIR  = '/mnt/data3/AirNow'
PLOTS_DIR = ROOT / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print(f"ROOT      : {ROOT}")
print(f"PLOTS_DIR : {PLOTS_DIR}")

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
from data.load_airnow import load_all, site_meta

df   = load_all(DATA_DIR)
meta = site_meta(DATA_DIR)

coverage = df.notna().mean()
mean_no2 = df.mean()

print(f"Shape  : {df.shape}  ({df.shape[0]} hours × {df.shape[1]} sites)")
print(f"Range  : {df.index[0]}  →  {df.index[-1]}")

## 1. Top-N individual sites

In [ ]:
# Top 5 sites by mean NO₂ (min 70 % coverage so we get full traces)
TOP_N   = 5
palette = plt.cm.tab10.colors
eligible = coverage[coverage >= 0.70].index
top_sites = mean_no2[eligible].nlargest(TOP_N)

fig, ax = plt.subplots(figsize=(14, 5))
for i, (site, _) in enumerate(top_sites.items()):
    daily = df[site].resample('D').mean().interpolate(method='time', limit=3)
    name  = meta.loc[site, 'name'] if site in meta.index else site
    ax.plot(daily.index, daily.values, lw=0.9, color=palette[i], label=name)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.set_xlabel('Date')
ax.set_ylabel('Daily mean NO₂ (PPB)')
ax.set_title(f'Top {TOP_N} highest-NO₂ sites (≥70 % coverage)')
ax.legend(fontsize=8)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'ts_top_sites.png', dpi=150)
plt.show()
print('Saved ts_top_sites.png')

## 2. Regional groupings — West / Central / East

In [ ]:
common = meta.index.intersection(df.columns)
meta_c = meta.loc[common]

regions = {
    'West  (lon < −105)':    meta_c[meta_c['lon'] < -105].index,
    'Central (−105 to −85)': meta_c[(meta_c['lon'] >= -105) & (meta_c['lon'] < -85)].index,
    'East  (lon ≥ −85)':     meta_c[meta_c['lon'] >= -85].index,
}
region_colors = {'West  (lon < −105)': 'steelblue',
                 'Central (−105 to −85)': 'darkorange',
                 'East  (lon ≥ −85)': 'seagreen'}

fig, ax = plt.subplots(figsize=(14, 5))
for region, sites in regions.items():
    sites_in_df = [s for s in sites if s in df.columns]
    if not sites_in_df:
        continue
    daily_region = df[sites_in_df].mean(axis=1).resample('D').mean()
    daily_region = daily_region.interpolate(method='time', limit=3)
    ax.plot(daily_region.index, daily_region.values,
            lw=1.3, color=region_colors[region],
            label=f'{region}  (n={len(sites_in_df)})')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.set_xlabel('Date')
ax.set_ylabel('Daily mean NO₂ (PPB)')
ax.set_title('Regional NO₂ time series — West / Central / East')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'ts_regional.png', dpi=150)
plt.show()
print('Saved ts_regional.png')

## 3. Seasonal overlays — average NO₂ by month of year

In [ ]:
# Average hourly NO₂ across all sites, grouped by month + hour → daily profile per month
network_hourly = df.mean(axis=1)

month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
season_colors = {
    'Jul': '#e6194b', 'Aug': '#f58231', 'Sep': '#ffe119',
    'Oct': '#3cb44b', 'Nov': '#4363d8', 'Dec': '#911eb4',
    'Jan': '#42d4f4', 'Feb': '#f032e6', 'Mar': '#bfef45',
    'Apr': '#fabed4', 'May': '#469990', 'Jun': '#dcbeff',
}

fig, ax = plt.subplots(figsize=(12, 5))
for m in range(1, 13):
    subset = network_hourly[network_hourly.index.month == m]
    diurnal = subset.groupby(subset.index.hour).mean()
    name    = month_names[m - 1]
    ax.plot(diurnal.index, diurnal.values, lw=1.5,
            color=season_colors[name], label=name, marker='o', markersize=3)

ax.set_xlabel('Hour of day (UTC)')
ax.set_ylabel('Mean NO₂ (PPB)')
ax.set_title('Seasonal diurnal overlay — average NO₂ by hour, split by month')
ax.set_xticks(range(0, 24, 2))
ax.legend(ncol=6, fontsize=8, loc='upper right')
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'ts_seasonal_overlay.png', dpi=150)
plt.show()
print('Saved ts_seasonal_overlay.png')

## 4. Rolling average / smoothed trends

In [ ]:
daily_all = df.mean(axis=1).resample('D').mean()

roll7  = daily_all.rolling(7,  center=True, min_periods=4).mean()
roll30 = daily_all.rolling(30, center=True, min_periods=15).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_all.index, daily_all.values,
        color='lightsteelblue', lw=0.7, alpha=0.8, label='Daily mean')
ax.plot(roll7.index,  roll7.values,
        color='steelblue', lw=1.4, label='7-day rolling mean')
ax.plot(roll30.index, roll30.values,
        color='darkred',   lw=2.0, label='30-day rolling mean')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.set_xlabel('Date')
ax.set_ylabel('NO₂ (PPB)')
ax.set_title('Network-wide NO₂ — daily, 7-day rolling, and 30-day rolling means')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'ts_rolling_avg.png', dpi=150)
plt.show()
print('Saved ts_rolling_avg.png')

## 5. Anomaly plot — deviation from the long-term daily mean

In [ ]:
# Anomaly = daily value minus the 30-day rolling baseline
baseline = daily_all.rolling(30, center=True, min_periods=15).mean()
anomaly  = daily_all - baseline

# Flag days more than 1.5× IQR above/below the anomaly distribution
q1, q3  = anomaly.quantile(0.25), anomaly.quantile(0.75)
iqr     = q3 - q1
pos_threshold =  1.5 * iqr
neg_threshold = -1.5 * iqr

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(anomaly.index, anomaly.values,
       color=np.where(anomaly > pos_threshold, 'crimson',
             np.where(anomaly < neg_threshold, 'steelblue', 'lightgrey')),
       width=1.0, label='Daily anomaly')
ax.axhline(pos_threshold, color='crimson',   lw=1.2, linestyle='--',
           label=f'+1.5×IQR threshold ({pos_threshold:.2f} ppb)')
ax.axhline(neg_threshold, color='steelblue', lw=1.2, linestyle='--',
           label=f'−1.5×IQR threshold ({neg_threshold:.2f} ppb)')
ax.axhline(0, color='black', lw=0.8)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.set_xlabel('Date')
ax.set_ylabel('Anomaly (PPB)')
ax.set_title('Network-wide NO₂ anomaly — deviation from 30-day rolling baseline\n'
             'Red = unusually high  |  Blue = unusually low  |  Grey = normal')
ax.legend(fontsize=8)
ax.grid(True, linestyle='--', alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'ts_anomaly.png', dpi=150)
plt.show()

n_high = (anomaly > pos_threshold).sum()
n_low  = (anomaly < neg_threshold).sum()
print(f'High-anomaly days : {n_high}')
print(f'Low-anomaly  days : {n_low}')
print('Saved ts_anomaly.png')

## 6. Anomaly detection overlay — spikes highlighted on the time series

In [ ]:
daily_ts  = df.mean(axis=1).resample('D').mean()
baseline  = daily_ts.rolling(30, center=True, min_periods=15).mean()
anomaly   = daily_ts - baseline

q1, q3    = anomaly.quantile(0.25), anomaly.quantile(0.75)
threshold = 1.5 * (q3 - q1)

high_idx = anomaly.index[anomaly >  threshold]
low_idx  = anomaly.index[anomaly < -threshold]

fig, ax = plt.subplots(figsize=(14, 5))

# Shade each anomalous day
for d in high_idx:
    ax.axvspan(d - pd.Timedelta('12h'), d + pd.Timedelta('12h'),
               color='crimson', alpha=0.18, linewidth=0)
for d in low_idx:
    ax.axvspan(d - pd.Timedelta('12h'), d + pd.Timedelta('12h'),
               color='steelblue', alpha=0.22, linewidth=0)

# Raw daily series + rolling baseline
ax.plot(daily_ts.index, daily_ts.values,
        color='#555555', lw=0.9, alpha=0.9, label='Daily mean NO₂')
ax.plot(baseline.index, baseline.values,
        color='black', lw=1.5, ls='--', label='30-day rolling baseline')

# Event markers
ax.scatter(high_idx, daily_ts.loc[high_idx],
           color='crimson', s=28, zorder=6, edgecolors='darkred', linewidths=0.5,
           label=f'High anomaly ({len(high_idx)} days)')
ax.scatter(low_idx, daily_ts.loc[low_idx],
           color='dodgerblue', s=28, zorder=6, edgecolors='navy', linewidths=0.5,
           label=f'Low anomaly  ({len(low_idx)} days)')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.set_xlabel('Date')
ax.set_ylabel('Daily mean NO₂ (PPB)')
ax.set_title(
    'Network-wide NO₂ with anomaly overlay  (threshold = ±1.5 × IQR from 30-day baseline)\n'
    'Red = unusually high  |  Blue = unusually low')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'ts_anomaly_overlay.png', dpi=150)
plt.show()
print(f'High anomaly days : {len(high_idx)}')
print(f'Low anomaly  days : {len(low_idx)}')
print('Saved ts_anomaly_overlay.png')
